# InternVL2微调实战
这里采用Xtuner工具微调InternVL2模型，使用LoRA微调视觉模块和文本模块，也可以选择对这两个模块进行量化。

## 模型下载
本项目中主要下载的是`1~8B`的模型
#### 多模态大语言模型 (InternVL 2.0)

<table>
  <tr>
    <th>Model Name</th>
    <th>Vision Part</th>
    <th>Language Part</th>
    <th>HF&nbsp;Link</th>
    <th>MS&nbsp;Link</th>
    <th>Document</th>
  </tr>
  <tr>
    <td>InternVL2&#8209;1B</td>
    <td><a href="https://huggingface.co/OpenGVLab/InternViT-300M-448px">InternViT&#8209;300M&#8209;448px</a></td>
    <td><a href="https://huggingface.co/Qwen/Qwen2-0.5B-Instruct">Qwen2&#8209;0.5B&#8209;Instruct</a></td>
    <td><a href="https://huggingface.co/OpenGVLab/InternVL2-1B">🤗 link</a></td>
    <td><a href="https://modelscope.cn/models/OpenGVLab/InternVL2-1B">🤖 link</a></td>
    <td><a href="https://internvl.readthedocs.io/en/latest/internvl2.0/introduction.html">📖 doc</a></td>
  </tr>
  <tr>
    <td>InternVL2&#8209;2B</td>
    <td><a href="https://huggingface.co/OpenGVLab/InternViT-300M-448px">InternViT&#8209;300M&#8209;448px</a></td>
    <td><a href="https://huggingface.co/internlm/internlm2-chat-1_8b">internlm2&#8209;chat&#8209;1&#8209;8b</a></td>
    <td><a href="https://huggingface.co/OpenGVLab/InternVL2-2B">🤗 link</a></td>
    <td><a href="https://modelscope.cn/models/OpenGVLab/InternVL2-2B">🤖 link</a></td>
    <td><a href="https://internvl.readthedocs.io/en/latest/internvl2.0/introduction.html">📖 doc</a></td>
  </tr>
  <tr>
    <td>InternVL2&#8209;4B</td>
    <td><a href="https://huggingface.co/OpenGVLab/InternViT-300M-448px">InternViT&#8209;300M&#8209;448px</a></td>
    <td><a href="https://huggingface.co/microsoft/Phi-3-mini-128k-instruct">Phi&#8209;3&#8209;mini&#8209;128k&#8209;instruct</a></td>
    <td><a href="https://huggingface.co/OpenGVLab/InternVL2-4B">🤗 link</a></td>
    <td><a href="https://modelscope.cn/models/OpenGVLab/InternVL2-4B">🤖 link</a></td>
    <td><a href="https://internvl.readthedocs.io/en/latest/internvl2.0/introduction.html">📖 doc</a></td>
  </tr>
  <tr>
    <td>InternVL2&#8209;8B</td>
    <td><a href="https://huggingface.co/OpenGVLab/InternViT-300M-448px">InternViT&#8209;300M&#8209;448px</a></td>
    <td><a href="https://huggingface.co/internlm/internlm2_5-7b-chat">internlm2_5&#8209;7b&#8209;chat</a></td>
    <td><a href="https://huggingface.co/OpenGVLab/InternVL2-8B">🤗 link</a></td>
    <td><a href="https://modelscope.cn/models/OpenGVLab/InternVL2-8B">🤖 link</a></td>
    <td><a href="https://internvl.readthedocs.io/en/latest/internvl2.0/introduction.html">📖 doc</a></td>
  </tr>
  
</table>

## 准备环境

这里我们来手动配置下xtuner。

### 配置虚拟环境

In [ ]:
conda create --name xtuner python=3.10 -y

# 激活虚拟环境（注意：后续的所有操作都需要在这个虚拟环境中进行）
conda activate xtuner

# 安装一些必要的库
conda install pytorch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 pytorch-cuda=12.1 -c pytorch -c nvidia -y
# 安装其他依赖
apt install libaio-dev
pip install transformers==4.39.3
pip install streamlit==1.36.0

### 安装xtuner

In [ ]:
# 创建一个目录，用来存放源代码
mkdir -p /root/InternLM/code

cd /root/InternLM/code

git clone -b v0.1.23  https://github.com/InternLM/XTuner

进入XTuner目录

In [ ]:
cd /root/InternLM/code/XTuner
pip install -e '.[deepspeed]'

# 安装验证
xtuner version
xtuner help

### 安装LMDeploy

pip install lmdeploy==0.5.3

### GraphViz安装

In [ ]:
pip install networkx[default]
pip install graphviz
apt install graphviz
apt-get update
apt-get install libgraphviz-dev
pip install pygraphviz
pip install pydot

## 准备微调数据集
### 数据集构造方式参考本项目中的InternVL实践指南_官方教程
[InternVL实践指南_官方教程](./InternVL实践指南_官方教程.md)

In [ ]:
[
    {
        "id": "000000033471",
        "image": ["coco/train2017/000000033471.jpg"], # 如果是纯文本，则该字段为 None 或者不存在
        "conversations": [
        {
            "from": "human",
            "value": "<image>\nWhat are the colors of the bus in the image?"
        },
        {
            "from": "gpt",
            "value": "The bus in the image is white and red."
        }
        ]
    }
]

## 配置微调参数
完整的微调参数见[internvl2_qlora_finetune.py](./internvl2_qlora_finetune.py)


#### Step 1 修改基本配置
必须修改的有：
- `path = '/root/models/InternVL2-8B'`: 模型权重位置
- `data_path = '/path/to/train_dataset.json'`: 微调的数据集
- `image_folder = '/path/to/img_folder'`：微调数据集中图片的保存目录

可调参数，注意这几个参数会影响内存占用，需要根据显卡资源灵活调整：
- `max_length = 8192`：输入的Embedding长度
- `batch_size = 4 # per_device `
- `accumulative_counts = 4`
- `dataloader_num_workers = 4`
- `max_epochs = 2`

修改保存checkpoints的步数：
- `save_steps = 100` ：每个多少步保存一次
- `save_total_limit = -1` ：保存的权重文件数量限制，-1为不限制，1则只保存一个权重文件，也就是只保存最新的一个

#### Step 2 修改模型配置
下面这个配置则主要是该如何训练模型。
- `freeze_llm`：是否冻结llm模块，为`True`则不训练。
- `freeze_visual_encoder`：是否冻结视觉模块，为`True`则不训练。
- `quantization_llm`：是否量化LLM
- `quantization_vit`：是否量化视觉模块
- `llm_lora`：LLM的LoRA配置
- `visual_encoder_lora`：视觉模块的LoRA配置

In [ ]:
model = dict(
    type=InternVL_V1_5,
    model_path=path,
    freeze_llm=True,
    freeze_visual_encoder=False,
    quantization_llm=False,  # or False
    quantization_vit=False,  # or True and uncomment visual_encoder_lora
    # comment the following lines if you don't want to use Lora in llm
    llm_lora=dict(
        type=LoraConfig,
        r=128,
        lora_alpha=256,
        lora_dropout=0.05,
        target_modules=None,
        task_type='CAUSAL_LM'),
    # uncomment the following lines if you don't want to use Lora in visual encoder # noqa
    # visual_encoder_lora=dict(
    #     type=LoraConfig, r=64, lora_alpha=16, lora_dropout=0.05,
    #     target_modules=['attn.qkv', 'attn.proj', 'mlp.fc1', 'mlp.fc2'])
)

其实保持不变即可。

## 开始训练
这里使用之前搞好的configs进行训练。咱们要调整一下batch size，并且使用qlora。要不半卡不够用的 QAQ。

#### 单机单卡训练

In [ ]:
CUDA_VISIBLE_DEVICES=1 NPROC_PER_NODE=1 xtuner train ./internvl_v2_internlm2_2b_qlora_finetune.py  --work-dir ./output_internvl/internvl_sft_flowchart2dot_v2  --deepspeed deepspeed_zero1

#### 单机多卡训练

In [ ]:
MKL_SERVICE_FORCE_INTEL=1 MKL_THREADING_LAYER=GNU NPROC_PER_NODE=2 xtuner train ./internvl_v2_internlm2_2b_qlora_finetune.py  --work-dir ./output_internvl/internvl_sft_flowchart2dot_v4  --deepspeed deepspeed_zero1

训练完成之后，在`./output_internvl/internvl_sft_flowchart`目录下会有一个pth文件，就算训练完成之后的权重。同时还有一个日志文件保存在`output_internvl/internvl_sft_flowchart/20240822_095431/20240822_095431.log`，这个日志的最后边输出模型权重保存的文件名。同时可以通过这个文件查看损失的下降情况。

## 合并权重&&模型转换
用官方脚本进行权重合并

> 将下面的iter_3000.pth切换成训练的饿到的pth即可。

In [ ]:
python3 ./convert_to_official.py ./internvl_v2_internlm2_2b_qlora_finetune.py ./output_internvl/internvl_sft_flowchart2json_v1/iter_1100.pth ./models/InternVL2-8B-flow2json_v1/

最后我们的模型在：`./models/InternVL2-2B/`，文件格式：

```text
.
|-- added_tokens.json
|-- config.json
|-- configuration_intern_vit.py
|-- configuration_internlm2.py
|-- configuration_internvl_chat.py
|-- conversation.py
|-- generation_config.json
|-- model.safetensors
|-- modeling_intern_vit.py
|-- modeling_internlm2.py
|-- modeling_internvl_chat.py
|-- special_tokens_map.json
|-- tokenization_internlm2.py
|-- tokenizer.model
`-- tokenizer_config.json
```